<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_EQVG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Setup and Initial Portfolio
# -----------------------------------------------------------------------------
# This cell loads the initial portfolio data and regulatory parameters. The output displays
# the starting positions, which corresponds to Step 1 in the report.

# Import necessary libraries
import pandas as pd
import numpy as np

# --- Initial Portfolio Data ---
# This data represents the starting positions and their gross vega sensitivities.
portfolio_data = [
    {'position_id': 1, 'bucket': 12, 'issuer': 'Issuer A', 'tenor_str': '10Y', 'gross_sensitivity': -18014},
    {'position_id': 2, 'bucket': 12, 'issuer': 'Issuer A', 'tenor_str': '10Y', 'gross_sensitivity': -366278},
    {'position_id': 3, 'bucket': 12, 'issuer': 'Issuer A', 'tenor_str': '6M', 'gross_sensitivity': -3958},
    {'position_id': 4, 'bucket': 12, 'issuer': 'Issuer A', 'tenor_str': '6M', 'gross_sensitivity': 14671},
    {'position_id': 5, 'bucket': 12, 'issuer': 'Issuer B', 'tenor_str': '10Y', 'gross_sensitivity': -102096},
    {'position_id': 6, 'bucket': 12, 'issuer': 'Issuer B', 'tenor_str': '6M', 'gross_sensitivity': 1040681}
]

# --- Regulatory Parameters ---
TENOR_TO_YEARS = {'6M': 0.5, '10Y': 10}
VEGA_RISK_WEIGHT = 0.7778
RHO_DELTA_DIFFERENT_ISSUER = 0.80
ALPHA = 0.01

# Create the initial DataFrame
df = pd.DataFrame(portfolio_data)

print("--- Step 1: Initial Portfolio and Risk Factors ---")
print("The calculation begins with the initial portfolio positions. Each unique combination")
print("of 'issuer' and 'tenor_str' represents a distinct risk factor.")
print("\n" + df[['position_id', 'bucket', 'issuer', 'tenor_str', 'gross_sensitivity']].to_string(index=False))


--- Step 1: Initial Portfolio and Risk Factors ---
The calculation begins with the initial portfolio positions. Each unique combination
of 'issuer' and 'tenor_str' represents a distinct risk factor.

 position_id  bucket   issuer tenor_str  gross_sensitivity
           1      12 Issuer A       10Y             -18014
           2      12 Issuer A       10Y            -366278
           3      12 Issuer A        6M              -3958
           4      12 Issuer A        6M              14671
           5      12 Issuer B       10Y            -102096
           6      12 Issuer B        6M            1040681


In [ ]:
# Cell 2: Step 3 - Net Sensitivities
# -----------------------------------------------------------------------------
# As per Article 325f(5), sensitivities for identical risk factors are netted.
# We group by the risk factor components and sum the sensitivities.
df_net = df.groupby(['bucket', 'issuer', 'tenor_str']).agg(
    net_sensitivity=('gross_sensitivity', 'sum')
).reset_index()
df_net['tenor_years'] = df_net['tenor_str'].map(TENOR_TO_YEARS)

print("--- Step 3: Net Sensitivities ---")
print("Gross sensitivities for each unique risk factor are summed to get the net sensitivity.")
print("\n" + df_net[['issuer', 'tenor_str', 'net_sensitivity']].to_string(index=False))


--- Step 3: Net Sensitivities ---
Gross sensitivities for each unique risk factor are summed to get the net sensitivity.

  issuer tenor_str  net_sensitivity
Issuer A       10Y          -384292
Issuer A        6M            10713
Issuer B       10Y          -102096
Issuer B        6M          1040681


In [ ]:
# Cell 3: Step 4 - Weighted Sensitivities
# -----------------------------------------------------------------------------
# Each net sensitivity is multiplied by the regulatory risk weight (77.78%).
df_weighted = df_net.copy()
df_weighted['weighted_sensitivity'] = df_weighted['net_sensitivity'] * VEGA_RISK_WEIGHT

# Calculate Sb for later steps
S_b = df_weighted['weighted_sensitivity'].sum()

print("--- Step 4: Weighted Sensitivities ---")
print("Net sensitivities are multiplied by the regulatory risk weight for the bucket.")
print("\n" + df_weighted[['issuer', 'tenor_str', 'net_sensitivity', 'weighted_sensitivity']].round(2).to_string(index=False))
print(f"\nSum of Weighted Sensitivities (S_12): {S_b:,.2f}")

--- Step 4: Weighted Sensitivities ---
Net sensitivities are multiplied by the regulatory risk weight for the bucket.

  issuer tenor_str  net_sensitivity  weighted_sensitivity
Issuer A       10Y          -384292            -298902.32
Issuer A        6M            10713               8332.57
Issuer B       10Y          -102096             -79410.27
Issuer B        6M          1040681             809441.68

Sum of Weighted Sensitivities (S_12): 439,461.67


In [ ]:
# Cell 4: Step 5 - Intra-Bucket Correlation Coefficients
# -----------------------------------------------------------------------------
# Per Article 325ay, correlations are calculated for each pair of risk factors.
positions = df_weighted.to_dict('records')
correlation_data = []
intra_corrs_medium = {}

for i in range(len(positions)):
    for j in range(i + 1, len(positions)):
        pos1, pos2 = positions[i], positions[j]

        pair_name = f"{pos1['issuer']}-{pos1['tenor_str']} vs {pos2['issuer']}-{pos2['tenor_str']}"
        pair_key = tuple(sorted((f"{pos1['issuer']}_{pos1['tenor_str']}", f"{pos2['issuer']}_{pos2['tenor_str']}")))

        rho_delta_comp = 1.0 if pos1['issuer'] == pos2['issuer'] else RHO_DELTA_DIFFERENT_ISSUER

        if pos1['tenor_years'] == pos2['tenor_years']:
            rho_maturity_comp = 1.0
        else:
            t1, t2 = pos1['tenor_years'], pos2['tenor_years']
            rho_maturity_comp = np.exp(-ALPHA * abs(t1 - t2) / min(t1, t2))

        rho_kl = rho_delta_comp * rho_maturity_comp
        intra_corrs_medium[pair_key] = rho_kl
        correlation_data.append([pair_name, f"{rho_delta_comp:.2%}", f"{rho_maturity_comp:.2%}", f"{rho_kl:.2%}"])

df_correlations = pd.DataFrame(correlation_data, columns=["Pair of Risk Factors", "Rho Delta", "Rho Maturity", "Final Correlation"])

print("--- Step 5: Intra-Bucket Correlation Coefficients (Medium Scenario) ---")
print("Correlations are determined based on issuer and option tenor similarity.")
print("\n" + df_correlations.to_string(index=False))

--- Step 5: Intra-Bucket Correlation Coefficients (Medium Scenario) ---
Correlations are determined based on issuer and option tenor similarity.

        Pair of Risk Factors Rho Delta Rho Maturity Final Correlation
 Issuer A-10Y vs Issuer A-6M   100.00%       82.70%            82.70%
Issuer A-10Y vs Issuer B-10Y    80.00%      100.00%            80.00%
 Issuer A-10Y vs Issuer B-6M    80.00%       82.70%            66.16%
 Issuer A-6M vs Issuer B-10Y    80.00%       82.70%            66.16%
  Issuer A-6M vs Issuer B-6M    80.00%      100.00%            80.00%
 Issuer B-10Y vs Issuer B-6M   100.00%       82.70%            82.70%


In [ ]:
# Cell 5: Step 7 - Intra-Bucket Aggregation (Medium Scenario)
# -----------------------------------------------------------------------------
# Calculate the bucket-specific capital charge (K_b) for the Medium Scenario.
positions_agg = df_weighted.to_dict('records')
sum_ws_sq = np.sum(df_weighted['weighted_sensitivity']**2)
cross_term = 0

for i in range(len(positions_agg)):
    for j in range(i + 1, len(positions_agg)):
        pos1, pos2 = positions_agg[i], positions_agg[j]
        key_part1 = f"{pos1['issuer']}_{pos1['tenor_str']}"
        key_part2 = f"{pos2['issuer']}_{pos2['tenor_str']}"
        pair_key = tuple(sorted((key_part1, key_part2)))
        rho_kl = intra_corrs_medium.get(pair_key, 0)
        cross_term += 2 * rho_kl * pos1['weighted_sensitivity'] * pos2['weighted_sensitivity']

K_b_medium = np.sqrt(max(0, sum_ws_sq + cross_term))

print("--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---")
print("Weighted sensitivities are aggregated using the specified correlations.")
print(f"\n1. Sum of Squares (Σ WS_k^2): {sum_ws_sq:,.2f}")
print(f"2. Sum of Cross-Products (Σ ρ_kl * WS_k * WS_l): {cross_term:,.2f}")
print("----------------------------------------------------------")
print(f"Bucket 12 Capital (K_12): {K_b_medium:,.2f}")

--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---
Weighted sensitivities are aggregated using the specified correlations.

1. Sum of Squares (Σ WS_k^2): 750,913,854,238.89
2. Sum of Cross-Products (Σ ρ_kl * WS_k * WS_l): -382,660,768,797.66
----------------------------------------------------------
Bucket 12 Capital (K_12): 606,838.60


In [ ]:
# Cell 6: Step 9 - Correlation Scenarios
# -----------------------------------------------------------------------------
# Recalculate the total capital for High and Low correlation scenarios.
def adjust_correlation(medium_corr, scenario):
    if scenario == 'High': return min(medium_corr * 1.25, 1.0)
    if scenario == 'Low': return max(2 * medium_corr - 1.0, 0.75 * medium_corr)
    return medium_corr

def calculate_capital(df_to_calc, corrs):
    sum_ws_sq = np.sum(df_to_calc['weighted_sensitivity']**2)
    cross_term = 0
    positions = df_to_calc.to_dict('records')
    for i in range(len(positions)):
        for j in range(i + 1, len(positions)):
            pos1, pos2 = positions[i], positions[j]
            key_part1 = f"{pos1['issuer']}_{pos1['tenor_str']}"
            key_part2 = f"{pos2['issuer']}_{pos2['tenor_str']}"
            pair_key = tuple(sorted((key_part1, key_part2)))
            rho_kl = corrs.get(pair_key, 0)
            cross_term += 2 * rho_kl * pos1['weighted_sensitivity'] * pos2['weighted_sensitivity']
    return np.sqrt(max(0, sum_ws_sq + cross_term))

high_intra = {k: adjust_correlation(v, 'High') for k, v in intra_corrs_medium.items()}
low_intra = {k: adjust_correlation(v, 'Low') for k, v in intra_corrs_medium.items()}

K_b_high = calculate_capital(df_weighted, high_intra)
K_b_low = calculate_capital(df_weighted, low_intra)

scenario_data = {
    'Scenario': ['Medium Correlation', 'High Correlation', 'Low Correlation'],
    'Equity Vega Capital Requirement': [K_b_medium, K_b_high, K_b_low]
}
df_scenarios = pd.DataFrame(scenario_data)

print("--- Step 9: Correlation Scenarios ---")
print("The capital is recalculated under stressed correlation assumptions.")
print("\nResulting Capital per Scenario:")
print(df_scenarios.round(2).to_string(index=False))

--- Step 9: Correlation Scenarios ---
The capital is recalculated under stressed correlation assumptions.

Resulting Capital per Scenario:
          Scenario  Equity Vega Capital Requirement
Medium Correlation                        606838.60
  High Correlation                        526391.44
   Low Correlation                        677803.97


In [ ]:
# Cell 7: Step 10 - Final Charge Calculation
# -----------------------------------------------------------------------------
# The final charge is the maximum of the three scenarios.
final_charge = df_scenarios['Equity Vega Capital Requirement'].max()
winning_scenario = df_scenarios.loc[df_scenarios['Equity Vega Capital Requirement'].idxmax()]['Scenario']

print("\n--- Step 10: Final Charge Calculation ---")
print(f"The final requirement is the maximum of the three scenarios.")
print("\n-------------------------------------------------")
print(f" Final Equity Vega Capital Requirement: {final_charge:,.2f}")
print(f" (Driven by the {winning_scenario})")
print("-------------------------------------------------")


--- Step 10: Final Charge Calculation ---
The final requirement is the maximum of the three scenarios.

-------------------------------------------------
 Final Equity Vega Capital Requirement: 677,803.97
 (Driven by the Low Correlation)
-------------------------------------------------
